# Computing Agent-Based Models: Dynamical Systems & Evolution

### Thomas Wildeboer | Apr 1. 2026

Department of Ecology and Evolutionary Biology

University of Toronto

---
<div>
<img src="img/bee.jpg" width="700"/>
</div>

Bee photo by Matthew T Rader (https://commons.wikimedia.org/wiki/File:Western_honey_bee_on_a_honeycomb.jpg)

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.colors import ListedColormap
import matplotlib.patches as mpatches
from scipy.integrate import solve_ivp

%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 100

---
## Outline

1. **What is Agent-Based Modeling?**
2. **Cellular Automata** — Conway's Game of Life
3. **Animal Behaviour** — The Boids Flocking Model
4. **Epidemiology** — SIR Models
5. **Evolutionary Genetics** — Adaptation in a changing environment
6. **Evolutionary Ecology** — Predator-Prey Coevolution

**Key idea:** Simple local rules → complex global behavior

---
## 1. What Is Agent-Based Modeling?

An **Agent-Based Model (ABM)**, often called an individual-based model in ecology, simulates a system as a collection of autonomous agents that:

- Have **state** (position, health, energy, genes, ...)
- Follow **rules** (move toward food, avoid predators, reproduce, transmit infection, ...)
- **Interact** with each other and their environment

---

### The Core Loop

```
Initialize agents
  └─> For each time step:
        ├─> Each agent observes local environment
        ├─> Each agent applies rules to update state
        ├─> Agents interact (collision, infection, mating, etc.)
        └─> Record global statistics
```

---

## 2. Cellular Automata: Conway's Game of Life

A simple ABM: agents are **grid cells**, rules are purely local.

### Rules (each cell looks at its 8 neighbors):
- **Live cell** with 2 or 3 live neighbors → **survives**
- **Live cell** with < 2 neighbors → **dies** (underpopulation)
- **Live cell** with > 3 neighbors → **dies** (overpopulation)
- **Dead cell** with exactly 3 neighbors → **born**

![Game of Life Rules](img/GoLDiagram.png)

These 4 rules produce: oscillators, gliders, still lifes, and chaotic growth.

---
### Starting Seeds:
![GoL seeds](img/GoLSeeds.jpg)

In [2]:
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import Image as IPImage

# Conway's Game of Life

def gol_step(grid):
    """Advance the grid by one generation using toroidal boundary conditions.

    Counts each cell's 8 neighbours by rolling the grid in all 8 directions
    and summing — a vectorised alternative to nested loops.
    Birth rule:   dead cell with exactly 3 neighbours → alive.
    Survival rule: live cell with 2 or 3 neighbours   → stays alive.
    """
    neighbour_count = sum(
        np.roll(np.roll(grid, row_shift, axis=0), col_shift, axis=1)
        for row_shift in [-1, 0, 1]
        for col_shift in [-1, 0, 1]
        if (row_shift, col_shift) != (0, 0)   # exclude the cell itself
    )
    return ((neighbour_count == 3) | (grid & (neighbour_count == 2))).astype(int)

GRID_SIZE = 60

def make_rpentomino():
    """R-pentomino: a 5-cell seed with rich, long-running dynamics."""
    grid = np.zeros((GRID_SIZE, GRID_SIZE), dtype=int)
    center_row, center_col = GRID_SIZE // 2, GRID_SIZE // 2
    for row_offset, col_offset in [(0, 1), (0, 2), (1, 0), (1, 1), (2, 1)]:
        grid[center_row + row_offset, center_col + col_offset] = 1
    return grid

def make_random(rng_seed=1234):
    """Uniformly random initial grid."""
    rng = np.random.default_rng(rng_seed)
    return rng.integers(0, 2, size=(GRID_SIZE, GRID_SIZE)).astype(int)

# Two initial conditions to compare
seeds = [('R-Pentomino', make_rpentomino()), ('Random (seed 1234)', make_random())]

# Pre-compute all frames, then animate
N_STEPS = 1000
all_frames = []
for label, initial_grid in seeds:
    grid = initial_grid.copy()
    run_frames = [grid.copy()]
    for _ in range(N_STEPS):
        grid = gol_step(grid)
        run_frames.append(grid.copy())
    all_frames.append(run_frames)

fig, axes = plt.subplots(1, 2, figsize=(11, 5), facecolor='black')
panel_images = []
for ax, (label, _) in zip(axes, seeds):
    ax.set_facecolor('black')
    ax.axis('off')
    im = ax.imshow(all_frames[0][0], cmap='hot', interpolation='nearest', vmin=0, vmax=1)
    ax.set_title(label, color='white', fontsize=10)
    panel_images.append(im)

step_label = fig.text(0.5, 0.01, '', ha='center', color='lightgray', fontsize=9)

def update_gol(frame):
    for im, run_frames in zip(panel_images, all_frames):
        im.set_data(run_frames[frame])
    live_cell_counts = [run[frame].sum() for run in all_frames]
    step_label.set_text(
        f'Step {frame}  |  Live cells: {live_cell_counts[0]}  vs  {live_cell_counts[1]}'
    )
    return panel_images + [step_label]

anim = FuncAnimation(fig, update_gol, frames=N_STEPS + 1, interval=60, blit=True)
anim.save('gol.gif', writer=PillowWriter(fps=15))
plt.close()


![Game of Life](img/gol.gif)

### Game of Life

- **Emergence**: Gliders, oscillators, and stable structures emerge from simple, local rules
- People have catalogued all sorts of 'species' in Game of Life
- **Turing complete**: it can compute any well-defined program (but so is Cities: Skylines)
- People have created a Game of Life program inside of Game of Life

---

## 3. Animal Behaviour: The Boids Flocking Model

Craig Reynolds (1986): agents move through space with **3 simple rules**.

| Rule | Description |
|------|-------------|
| **Separation** | Avoid crowding neighbors |
| **Alignment** | Steer toward average heading of neighbors |
| **Cohesion** | Steer toward average position of neighbors |

Agents are now points in space.

No global coordination, flock behavior is **emergent** from local rules.

In [3]:
# Boids Flocking (Reynolds 1987) vs Random Walk

class Boid:
    """A single boid agent with 2D position and velocity."""
    def __init__(self, x, y, vx, vy):
        self.pos = np.array([x, y], dtype=float)
        self.vel = np.array([vx, vy], dtype=float)


class BoidsSimulation:
    """
    Reynolds' three-rule flocking model on a toroidal 2D plane.

    At each step every boid applies three steering forces:
      1. Separation: repel from boids within sep_radius  (avoid crowding)
      2. Alignment: match average velocity within align_radius
      3. Cohesion: steer toward average position within coh_radius
    """
    def __init__(self, n=80, width=100, height=100,
                 sep_radius=4, align_radius=15, coh_radius=15,
                 sep_weight=1.5, align_weight=1.0, coh_weight=1.0,
                 max_speed=2.5, max_force=0.1):
        self.width        = width
        self.height       = height
        self.sep_radius   = sep_radius
        self.align_radius = align_radius
        self.coh_radius   = coh_radius
        self.sep_weight   = sep_weight
        self.align_weight = align_weight
        self.coh_weight   = coh_weight
        self.max_speed    = max_speed
        self.max_force    = max_force
        rng = np.random.default_rng(123)
        self.boids = [
            Boid(rng.uniform(0, width), rng.uniform(0, height),
                 rng.uniform(-1, 1),    rng.uniform(-1, 1))
            for _ in range(n)
        ]

    def _clamp_magnitude(self, vector, max_magnitude):
        """Scale vector down to max_magnitude if it exceeds it; otherwise unchanged."""
        magnitude = np.linalg.norm(vector)
        return vector / magnitude * max_magnitude if magnitude > max_magnitude else vector

    def _steering_force(self, desired_direction, current_velocity):
        """
        Steering: compute a force that nudges current_velocity toward
        desired_direction at max_speed, clamped to max_force.
        """
        magnitude = np.linalg.norm(desired_direction)
        if magnitude == 0:
            return np.zeros(2)
        target_velocity = (desired_direction / magnitude) * self.max_speed
        raw_force = target_velocity - current_velocity
        return self._clamp_magnitude(raw_force, self.max_force)

    def step(self):
        # Snapshot positions/velocities so all boids react to the same world state
        positions  = np.array([b.pos for b in self.boids])
        velocities = np.array([b.vel for b in self.boids])

        for i, boid in enumerate(self.boids):
            displacements = positions - boid.pos           # vectors to every other boid
            distances     = np.linalg.norm(displacements, axis=1)
            distances[i]  = np.inf                         # exclude self

            # Separation:
            sep_mask = distances < self.sep_radius
            if sep_mask.any():
                # Repulsion weighted by 1/dist^2
                weighted_repulsion = -np.sum(
                    displacements[sep_mask] / (distances[sep_mask, None] ** 2 + 1e-9),
                    axis=0
                )
                sep_force = self._steering_force(weighted_repulsion, boid.vel)
            else:
                sep_force = np.zeros(2)

            # Alignment:
            align_mask = (distances < self.align_radius) & ~sep_mask
            if align_mask.any():
                avg_flock_velocity = velocities[align_mask].mean(axis=0)
                align_force = self._steering_force(avg_flock_velocity, boid.vel)
            else:
                align_force = np.zeros(2)

            # Cohesion:
            coh_mask = (distances < self.coh_radius) & ~sep_mask
            if coh_mask.any():
                direction_to_centre = positions[coh_mask].mean(axis=0) - boid.pos
                coh_force = self._steering_force(direction_to_centre, boid.vel)
            else:
                coh_force = np.zeros(2)

            boid.vel += (self.sep_weight   * sep_force
                       + self.align_weight * align_force
                       + self.coh_weight   * coh_force)
            boid.vel = self._clamp_magnitude(boid.vel, self.max_speed)
            boid.pos = (boid.pos + boid.vel) % np.array([self.width, self.height])


class RWSimulation:
    """Agents moving at constant speed with a randomly-perturbed heading each step."""
    def __init__(self, n=80, width=100, height=100, speed=2.5):
        self.width  = width
        self.height = height
        self.speed  = speed
        rng = np.random.default_rng(123)
        self.pos = rng.uniform([0, 0], [width, height], size=(n, 2))
        initial_angles = rng.uniform(0, 2 * np.pi, n)
        self.vel = np.stack([np.cos(initial_angles), np.sin(initial_angles)], axis=1) * speed
        self._rng = rng

    def step(self):
        # Add Gaussian noise to each agent's heading, keep speed constant
        headings  = np.arctan2(self.vel[:, 1], self.vel[:, 0])
        headings += self._rng.normal(0, 0.4, len(headings))
        self.vel  = np.stack([np.cos(headings), np.sin(headings)], axis=1) * self.speed
        self.pos  = (self.pos + self.vel) % np.array([self.width, self.height])


N_AGENTS     = 200
boids_sim    = BoidsSimulation(n=N_AGENTS)
rw_sim       = RWSimulation(n=N_AGENTS)

fig, (ax_boids, ax_rw) = plt.subplots(1, 2, figsize=(13, 6), facecolor='#0d0d2b')
for ax, title in [(ax_boids, 'Boids'), (ax_rw, 'Random Walk')]:
    ax.set_facecolor('#0d0d2b'); ax.axis('off')
    ax.set_xlim(0, 100); ax.set_ylim(0, 100)
    ax.set_title(title, color='white', fontsize=11)

# Quiver: arrow position = agent location, direction = velocity, colour = speed
initial_pos = np.array([b.pos for b in boids_sim.boids])
initial_vel = np.array([b.vel for b in boids_sim.boids])
quiver_boids = ax_boids.quiver(
    initial_pos[:, 0], initial_pos[:, 1],
    initial_vel[:, 0], initial_vel[:, 1],
    np.linalg.norm(initial_vel, axis=1),
    cmap='cool', scale=60, width=0.005, clim=(0, 2.5)
)
quiver_rw = ax_rw.quiver(
    rw_sim.pos[:, 0], rw_sim.pos[:, 1],
    rw_sim.vel[:, 0], rw_sim.vel[:, 1],
    np.linalg.norm(rw_sim.vel, axis=1),
    cmap='autumn', scale=60, width=0.005, clim=(0, 2.5)
)

step_label = fig.text(0.5, 0.02, 'Step 0', ha='center', color='lightgray', fontsize=9)

def update_boids_animation(frame):
    boids_sim.step()
    rw_sim.step()

    boids_pos = np.array([b.pos for b in boids_sim.boids])
    boids_vel = np.array([b.vel for b in boids_sim.boids])
    quiver_boids.set_offsets(boids_pos)
    quiver_boids.set_UVC(boids_vel[:, 0], boids_vel[:, 1], np.linalg.norm(boids_vel, axis=1))

    quiver_rw.set_offsets(rw_sim.pos)
    quiver_rw.set_UVC(rw_sim.vel[:, 0], rw_sim.vel[:, 1],
                            np.linalg.norm(rw_sim.vel, axis=1))

    step_label.set_text(f'Step {frame + 30}')
    return quiver_boids, quiver_rw, step_label

anim = FuncAnimation(fig, update_boids_animation, frames=300, interval=50, blit=True)
anim.save('boids.gif', writer=PillowWriter(fps=20))
plt.close()


![Boids](img/boids.gif)

---

## 4. Epidemiology: SIR Model

Classic compartmental model in epidemiology

Compartmental model: agents are assigned one of 3 states:

### States
- **S** = Susceptible
- **I** = Infectious  
- **R** = Recovered (immune)

Agents move between states depending on local rules and environment.

### Basic model:
$$\frac{dS}{dt} = -\beta SI$$
$$\frac{dI}{dt} = \beta SI - \gamma I$$
$$\frac{dR}{dt} = \gamma I$$

Easy to solve numerically from initial conditions:

In [4]:
# Numerical solution of SIR ODE

def sir_ode(t, y, beta, gamma):
    S, I, R = y
    N = S + I + R                     
    new_infections = beta * S * I / N
    new_recoveries = gamma * I
    dS = -new_infections
    dI =  new_infections - new_recoveries
    dR =  new_recoveries
    return [dS, dI, dR]

population_size = 1000
beta  = 0.3           # transmission rate
gamma = 0.05          # recovery rate  (mean infectious period = 1/gamma = 20 time units)
R0    = beta / gamma  # basic reproduction number

ode_solution = solve_ivp(
    sir_ode,
    t_span=[0, 200],
    y0=[population_size - 1, 1, 0], # start with 1 infectious individual
    args=(beta, gamma),
    t_eval=np.linspace(0, 200, 1000)
)

In [5]:
fig, ax = plt.subplots(figsize=(9, 4), facecolor='#111')
ax.set_facecolor('#0d0d1a')
ax.plot(ode_solution.t, ode_solution.y[0], color='#4c9be8', lw=2.5, label='S (Susceptible)')
ax.plot(ode_solution.t, ode_solution.y[1], color='#e84c4c', lw=2.5, label='I (Infectious)')
ax.plot(ode_solution.t, ode_solution.y[2], color='#4ce87a', lw=2.5, label='R (Recovered)')
ax.set_xlabel('Time', color='white')
ax.set_ylabel('Population', color='white')
ax.set_title(f'SIR ODE Solution (β={beta}, γ={gamma}, R₀={R0:.1f})', color='white', fontsize=12)
ax.tick_params(colors='white')
for spine in ax.spines.values():
    spine.set_color('#444')
ax.legend(facecolor='#222', labelcolor='white', fontsize=10)
ax.grid(True, alpha=0.12, color='#444')
plt.tight_layout()
plt.savefig('SIR_ODE.png')
plt.close()

![SIR solution](img/SIR_ODE.png)

### Spatial structure:

Individuals live on a 2D plane and can only infect those within a specified radius.

Easy to implement as an ABM:

In [6]:
# Spatial SIR: epidemic spreading through local contact
# Agents are stationary; infection only occurs within radius `rad`.

class SIRAgent:
    S, I, R = 0, 1, 2   # state constants
    def __init__(self, x, y, state=0):
        self.pos   = np.array([x, y], dtype=float)
        self.state = state

class SpatialSIR:
    def __init__(self, n=1500, width=80, beta=0.3, gamma=0.05, rad=3):
        self.width = width
        self.beta  = beta   # per-contact transmission probability
        self.gamma = gamma  # per-step recovery probability
        self.rad   = rad    # infection radius
        rng = np.random.default_rng(0)
        self.agents = [
            SIRAgent(rng.uniform(0, width), rng.uniform(0, width))
            for _ in range(n)
        ]
        for agent in self.agents[:4]:   # seed with 4 initial infectious agents
            agent.state = SIRAgent.I

    def step(self):
        # Snapshot positions and states (so all agents react to the same world)
        all_positions = np.array([a.pos for a in self.agents])
        all_states    = np.array([a.state for a in self.agents])
        for agent in self.agents:
            if agent.state == SIRAgent.S:
                distances = np.linalg.norm(all_positions - agent.pos, axis=1)
                n_infectious_neighbors = np.sum(
                    (distances < self.rad) & (all_states == SIRAgent.I)
                )
                # Each infectious neighbour independently attempts transmission
                if n_infectious_neighbors > 0:
                    if np.random.rand() < 1 - (1 - self.beta) ** n_infectious_neighbors:
                        agent.state = SIRAgent.I
            elif agent.state == SIRAgent.I:
                if np.random.rand() < self.gamma:
                    agent.state = SIRAgent.R

    def counts(self):
        states = np.array([a.state for a in self.agents])
        return (states == 0).sum(), (states == 1).sum(), (states == 2).sum()

In [7]:
sir_sim = SpatialSIR()
STATE_COLORS = {0: '#4c9be8', 1: '#e84c4c', 2: '#4ce87a'}

fig, (ax_spatial, ax_timeseries) = plt.subplots(1, 2, figsize=(11, 5), facecolor='#111')
for ax in (ax_spatial, ax_timeseries):
    ax.set_facecolor('#111')
ax_spatial.axis('off')
ax_spatial.set_xlim(0, sir_sim.width); ax_spatial.set_ylim(0, sir_sim.width)

initial_positions = np.array([a.pos for a in sir_sim.agents])
initial_colors    = [STATE_COLORS[a.state] for a in sir_sim.agents]
agent_scatter     = ax_spatial.scatter(
    initial_positions[:, 0], initial_positions[:, 1],
    c=initial_colors, s=8, linewidths=0
)
ax_spatial.set_title('Spatial SIR', color='white', fontsize=11)

timesteps, s_history, i_history, r_history = [], [], [], []
s_line, = ax_timeseries.plot([], [], color='#4c9be8', linewidth=2, label='S')
i_line, = ax_timeseries.plot([], [], color='#e84c4c', linewidth=2, label='I')
r_line, = ax_timeseries.plot([], [], color='#4ce87a', linewidth=2, label='R')
ax_timeseries.set_xlim(0, 150); ax_timeseries.set_ylim(0, len(sir_sim.agents))
ax_timeseries.legend(facecolor='#222', labelcolor='white', fontsize=9)
ax_timeseries.tick_params(colors='white')
ax_timeseries.set_xlabel('Time', color='white')
for spine in ax_timeseries.spines.values():
    spine.set_color('#444')

def update_sir(frame):
    sir_sim.step()
    agent_scatter.set_color([STATE_COLORS[a.state] for a in sir_sim.agents])
    n_s, n_i, n_r = sir_sim.counts()
    timesteps.append(frame)
    s_history.append(n_s); i_history.append(n_i); r_history.append(n_r)
    s_line.set_data(timesteps, s_history)
    i_line.set_data(timesteps, i_history)
    r_line.set_data(timesteps, r_history)
    return agent_scatter, s_line, i_line, r_line

anim = FuncAnimation(fig, update_sir, frames=150, interval=60, blit=True)
anim.save('sir.gif', writer=PillowWriter(fps=18))
plt.close()

![Spatial SIR](img/sir.gif)

---

### Spatial SIR with moving and flocking agents

Real populations **cluster** — households, workplaces, social groups — violating this assumption.

**RW-SIR ABM**: agents move at a constant velocity in a random heading at each time step (random walk).

**Boids-SIR ABM**: agents move with flocking rules (separation + alignment + cohesion) instead of random walks.

Identical epidemic parameters: $\beta = 0.30$, $\gamma = 0.05$, infection radius $r = 5.0$. Only movement rules change.

| | Random Walk | Boids Flocking |
|---|---|---|
| Contact structure | Homogeneous | Heterogeneous |
| Within-group contact | Moderate, transient | High, sustained |
| Between-group contact | Frequent | Rare |
| Epidemic shape | Flat | Pulsed |


In [8]:
# Boids-SIR vs Random-Walk SIR

N_AGENTS    = 200
WORLD_WIDTH = 100
BETA        = 0.30   # per-contact transmission probability
GAMMA       = 0.05   # per-step recovery probability
INF_RADIUS  = 5.0    # infection radius (same in both models)
N_STEPS     = 200

# RW SIR
class RandWalkSIR:
    S, I, R = 0, 1, 2

    def __init__(self, n=200, width=100, beta=0.3, gamma=0.05, rad=5.0, seed=0):
        rng = np.random.default_rng(seed)
        self.n     = n
        self.width = width
        self.beta  = beta
        self.gamma = gamma
        self.rad   = rad
        self.pos   = rng.uniform(0, width, (n, 2))
        self.vel   = rng.uniform(-0.5, 0.5, (n, 2))
        self.state = np.zeros(n, dtype=int)
        self.state[:4] = self.I

    def step(self):
        W = self.width
        # Move: random-walk velocity perturbation, clamped to [-0.8, 0.8]
        self.vel += np.random.uniform(-0.2, 0.2, (self.n, 2))
        self.vel  = np.clip(self.vel, -0.8, 0.8)
        self.pos  = (self.pos + self.vel) % W

        # Pairwise toroidal distances: d[i,j] = shortest-path displacement from i to j
        displacement = ((self.pos[np.newaxis] - self.pos[:, np.newaxis] + W/2) % W) - W/2
        distances    = np.sqrt((displacement ** 2).sum(axis=2))
        np.fill_diagonal(distances, np.inf)

        # Count infectious neighbours within radius for each agent
        n_infectious_neighbors = (
            (distances < self.rad) & (self.state == self.I)[np.newaxis]
        ).sum(axis=1)

        # Vectorised state transitions (evaluated on the same snapshot)
        new_infected  = ((self.state == self.S)
                         & (n_infectious_neighbors > 0)
                         & (np.random.rand(self.n) < 1 - (1 - self.beta) ** n_infectious_neighbors))
        new_recovered = (self.state == self.I) & (np.random.rand(self.n) < self.gamma)
        self.state[new_infected]  = self.I
        self.state[new_recovered] = self.R

    def counts(self):
        return (self.state == 0).sum(), (self.state == 1).sum(), (self.state == 2).sum()


# Boids SIR
class BoidsSIR:
    S, I, R = 0, 1, 2

    def __init__(self, n=200, width=100, beta=0.3, gamma=0.05, rad=3.0,
                 sep_radius=3.0, align_radius=12.0, coh_radius=12.0,
                 sep_weight=1.5, align_weight=1.0, coh_weight=1.0,
                 max_speed=1.5, max_force=0.1, seed=0):
        rng = np.random.default_rng(seed)
        self.n            = n
        self.width        = width
        self.beta         = beta
        self.gamma        = gamma
        self.rad          = rad
        self.sep_radius   = sep_radius
        self.align_radius = align_radius
        self.coh_radius   = coh_radius
        self.sep_weight   = sep_weight
        self.align_weight = align_weight
        self.coh_weight   = coh_weight
        self.max_speed    = max_speed
        self.max_force    = max_force
        self.pos   = rng.uniform(0, width, (n, 2))
        angles     = rng.uniform(0, 2 * np.pi, n)
        self.vel   = np.stack([np.cos(angles), np.sin(angles)], axis=1) * (max_speed * 0.5)
        self.state = np.zeros(n, dtype=int)
        self.state[:4] = self.I

    def _steering_force(self, raw_direction):
        """Vectorised Reynolds steering for all n agents simultaneously.

        raw_direction: (n, 2) desired movement direction (unnormalised).
        Returns:       (n, 2) steering force, clamped to max_force.
        Agents with no neighbours (zero raw_direction) get zero force.
        """
        magnitude = np.sqrt((raw_direction ** 2).sum(axis=1, keepdims=True))
        has_neighbors = magnitude > 1e-6
        # Normalise, scale to max_speed, subtract current velocity → steering force
        target_velocity = np.where(has_neighbors, raw_direction / (magnitude + 1e-9), 0.0) * self.max_speed
        force     = target_velocity - self.vel
        force_mag = np.sqrt((force ** 2).sum(axis=1, keepdims=True)) + 1e-9
        force     = np.where(force_mag > self.max_force, force / force_mag * self.max_force, force)
        return np.where(has_neighbors, force, 0.0)   # zero force when no neighbours

    def step(self):
        W = self.width
        # Toroidal pairwise displacement: displacement[i, j] = pos[j] - pos[i]
        displacement = ((self.pos[np.newaxis] - self.pos[:, np.newaxis] + W/2) % W) - W/2
        distances = np.sqrt((displacement ** 2).sum(axis=2))
        np.fill_diagonal(distances, np.inf)

        sep_mask   = distances < self.sep_radius
        align_mask = (distances < self.align_radius) & ~sep_mask
        coh_mask   = (distances < self.coh_radius)   & ~sep_mask

        # Separation: steer away from too-close neighbours, weighted by 1/dist²
        sep_direction = -(displacement * (sep_mask / (distances ** 2 + 1e-9))[:, :, None]).sum(axis=1)

        # Alignment: steer to match average velocity of flock-mates
        align_count   = align_mask.sum(axis=1)
        align_vel_sum = (align_mask[:, :, None] * self.vel[np.newaxis]).sum(axis=1)
        align_direction = np.where(
            align_count[:, None] > 0, align_vel_sum / (align_count[:, None] + 1e-9), 0.0
        )

        # Cohesion: steer toward average position of flock-mates
        coh_count   = coh_mask.sum(axis=1)
        coh_disp_sum = (coh_mask[:, :, None] * displacement).sum(axis=1)
        coh_direction = np.where(
            coh_count[:, None] > 0, coh_disp_sum / (coh_count[:, None] + 1e-9), 0.0
        )

        self.vel += (self.sep_weight   * self._steering_force(sep_direction)
                   + self.align_weight * self._steering_force(align_direction)
                   + self.coh_weight   * self._steering_force(coh_direction))
        speed = np.sqrt((self.vel ** 2).sum(axis=1, keepdims=True))
        self.vel = np.where(speed > self.max_speed, self.vel / speed * self.max_speed, self.vel)
        self.pos = (self.pos + self.vel) % W

        # Infection using updated (post-movement) positions
        disp2     = ((self.pos[np.newaxis] - self.pos[:, np.newaxis] + W/2) % W) - W/2
        dist2     = np.sqrt((disp2 ** 2).sum(axis=2))
        np.fill_diagonal(dist2, np.inf)
        n_infectious_neighbors = (
            (dist2 < self.rad) & (self.state == self.I)[np.newaxis]
        ).sum(axis=1)
        new_infected  = ((self.state == self.S)
                         & (n_infectious_neighbors > 0)
                         & (np.random.rand(self.n) < 1 - (1 - self.beta) ** n_infectious_neighbors))
        new_recovered = (self.state == self.I) & (np.random.rand(self.n) < self.gamma)
        self.state[new_infected]  = self.I
        self.state[new_recovered] = self.R

    def counts(self):
        return (self.state == 0).sum(), (self.state == 1).sum(), (self.state == 2).sum()


randwalk_sim  = RandWalkSIR(n=N_AGENTS, width=WORLD_WIDTH, beta=BETA, gamma=GAMMA, rad=INF_RADIUS)
boids_sir_sim = BoidsSIR(n=N_AGENTS, width=WORLD_WIDTH, beta=BETA, gamma=GAMMA, rad=INF_RADIUS)

# Animations:
SIR_COLORS = {0: '#4c9be8', 1: '#e84c4c', 2: '#4ce87a'}
fig      = plt.figure(figsize=(14, 9), facecolor='#111')
grid_spec = fig.add_gridspec(2, 2, hspace=0.42, wspace=0.32)
ax_randwalk_spatial = fig.add_subplot(grid_spec[0, 0])
ax_boids_spatial    = fig.add_subplot(grid_spec[0, 1])
ax_randwalk_ts      = fig.add_subplot(grid_spec[1, 0])
ax_boids_ts         = fig.add_subplot(grid_spec[1, 1])

for ax, title in [(ax_randwalk_spatial, 'Random Walk SIR'), (ax_boids_spatial, 'Boids SIR')]:
    ax.set_facecolor('#111'); ax.axis('off')
    ax.set_xlim(0, WORLD_WIDTH); ax.set_ylim(0, WORLD_WIDTH)
    ax.set_title(title, color='white', fontsize=11)

for ax, title in [(ax_randwalk_ts, 'Random Walk SIR'), (ax_boids_ts, 'Boids SIR')]:
    ax.set_facecolor('#0d0d1a'); ax.tick_params(colors='white')
    for spine in ax.spines.values(): spine.set_color('#444')
    ax.set_xlim(0, N_STEPS); ax.set_ylim(0, N_AGENTS)
    ax.set_xlabel('Step', color='white'); ax.set_ylabel('Count', color='white')
    ax.set_title(title, color='white', fontsize=10)

legend_proxies = [plt.Line2D([0], [0], color=c, lw=2)
                  for c in ['#4c9be8', '#e84c4c', '#4ce87a']]
for ax in [ax_randwalk_ts, ax_boids_ts]:
    ax.legend(legend_proxies, ['S', 'I', 'R'], facecolor='#222', labelcolor='white', fontsize=9)

scatter_randwalk = ax_randwalk_spatial.scatter(
    *randwalk_sim.pos.T, c=[SIR_COLORS[s] for s in randwalk_sim.state], s=8)
scatter_boids    = ax_boids_spatial.scatter(
    *boids_sir_sim.pos.T, c=[SIR_COLORS[s] for s in boids_sir_sim.state], s=8)
randwalk_lines   = {k: ax_randwalk_ts.plot([], [], color=SIR_COLORS[i], lw=2)[0]
                   for i, k in enumerate('SIR')}
boids_lines      = {k: ax_boids_ts.plot([], [], color=SIR_COLORS[i], lw=2)[0]
                   for i, k in enumerate('SIR')}
timesteps        = []
randwalk_history = {'S': [], 'I': [], 'R': []}
boids_history    = {'S': [], 'I': [], 'R': []}

def update_sir_comparison(frame):
    randwalk_sim.step(); boids_sir_sim.step()
    scatter_randwalk.set_offsets(randwalk_sim.pos)
    scatter_randwalk.set_color([SIR_COLORS[s] for s in randwalk_sim.state])
    scatter_boids.set_offsets(boids_sir_sim.pos)
    scatter_boids.set_color([SIR_COLORS[s] for s in boids_sir_sim.state])
    timesteps.append(frame + 1)
    for history, sim in [(randwalk_history, randwalk_sim), (boids_history, boids_sir_sim)]:
        n_s, n_i, n_r = sim.counts()
        history['S'].append(n_s); history['I'].append(n_i); history['R'].append(n_r)
    for k in 'SIR':
        randwalk_lines[k].set_data(timesteps, randwalk_history[k])
        boids_lines[k].set_data(timesteps, boids_history[k])
    return scatter_randwalk, scatter_boids, *randwalk_lines.values(), *boids_lines.values()

anim = FuncAnimation(fig, update_sir_comparison, frames=N_STEPS, interval=50, blit=True)
anim.save('sir_boids.gif', writer=PillowWriter(fps=20))
plt.close()

![SIR Boids vs RW](img/sir_boids.gif)

---

## 5. Evolutionary Genetics: Adaptation in a changing environment

We model a population of agents with **heritable traits** under selection for some optimal value that changes with time.

### Model
- Agents have a **genome**: a vector of real-valued traits
- **Fitness** = how well traits match the environment's optimum
- Discrete generations: selection → reproduction with mutation → new population
- Environment can **shift** to study adaptation

![Lifecycle](img/lifecycle.png)

In [9]:
# Population tracking a moving fitness optimum

class EvolutionSim:
    """Asexual population with a heritable trait evolving under Gaussian selection.

    Each generation:
      1. Each individual's fitness is f(x) = exp(-(x - optimum)^2 / (2*sel_strength^2))
      2. Each individual produces Poisson(fitness * birth_rate) offspring
      3. Each offspring inherits the parent's trait, then mutates with probability mut_rate
      4. If the population exceeds max_pop, a random subset is kept (density regulation)
    """
    def __init__(self, pop_size=300, mut_rate=0.1, mut_std=0.5,
                 sel_strength=1.5, birth_rate=1.3, max_pop=500):
        rng = np.random.default_rng(1)
        self.traits       = rng.normal(0, 0.8, pop_size)
        self.optimum      = 0.0
        self.mut_rate     = mut_rate
        self.mut_std      = mut_std
        self.sel_strength = sel_strength
        self.birth_rate   = birth_rate   # expected offspring per individual at peak fitness
        self.max_pop      = max_pop
        self.extinct      = False

    def fitness(self):
        """Gaussian fitness centred on the current optimum."""
        return np.exp(-(self.traits - self.optimum) ** 2 / (2 * self.sel_strength ** 2))

    def step(self, shift=None):
        if self.extinct:
            return 0, 0.0
        if shift is not None:
            self.optimum += shift
        fitness_values = self.fitness()
        mean_fitness   = float(fitness_values.mean())
        
        # Each individual independently produces a Poisson-distributed number of offspring
        offspring_counts = np.random.poisson(fitness_values * self.birth_rate)
        offspring_traits = np.repeat(self.traits, offspring_counts)
        if len(offspring_traits) == 0:
            self.extinct = True
            self.traits  = np.array([])
            return 0, mean_fitness
            
        # Mutation: random fraction of offspring receive a Gaussian trait perturbation
        mutation_mask = np.random.rand(len(offspring_traits)) < self.mut_rate
        offspring_traits[mutation_mask] += np.random.normal(0, self.mut_std, mutation_mask.sum())
        
        # Density regulation: cap population by random culling
        if len(offspring_traits) > self.max_pop:
            offspring_traits = np.random.choice(offspring_traits, self.max_pop, replace=False)
            
        self.traits = offspring_traits
        
        return len(self.traits), mean_fitness

# Let the population reach equilibrium before introducing the moving optimum
evolution_sim = EvolutionSim(pop_size=300)
for _ in range(30):
    evolution_sim.step()

TRAIT_LIM     = 30
shift_per_gen = 0.1   # optimum moves this far each generation
trait_snapshots, optimum_log, pop_size_log, mean_fitness_log = [], [], [], []

for generation in range(300):
    shift = shift_per_gen if generation >= 10 else None   # stable for first 10 gens
    pop_size, mean_fitness = evolution_sim.step(shift=shift)
    trait_snapshots.append(evolution_sim.traits.copy())
    optimum_log.append(evolution_sim.optimum)
    pop_size_log.append(pop_size)
    mean_fitness_log.append(mean_fitness)

# ── Figure: trait distribution histogram | mean fitness over time ─────────────
fig = plt.figure(figsize=(10, 5), facecolor='#111')
ax_histogram   = fig.add_subplot(1, 2, 1)
ax_mean_fitness = fig.add_subplot(1, 2, 2)

for ax in (ax_histogram, ax_mean_fitness):
    ax.set_facecolor('#111')
    for spine in ax.spines.values(): spine.set_color('#444')
    ax.tick_params(colors='white')

ax_histogram.set_xlim(-5, TRAIT_LIM); ax_histogram.set_ylim(0, 120)
ax_histogram.set_xlabel('Trait value', color='white')
ax_histogram.set_ylabel('Count', color='white')
histogram_title = ax_histogram.set_title('Gen 0 — stable environment', color='white', fontsize=11)

trait_bins     = np.linspace(-5, TRAIT_LIM, 40)
histogram_bars = ax_histogram.bar(
    trait_bins[:-1], np.zeros(len(trait_bins) - 1),
    width=trait_bins[1] - trait_bins[0], color='#9b59b6', alpha=0.8, align='edge'
)
optimum_line = ax_histogram.axvline(
    optimum_log[0], color='#00ffcc', linewidth=2, linestyle='--', label='Optimum'
)
ax_histogram.legend(facecolor='#222', labelcolor='white', fontsize=8)

fitness_line, = ax_mean_fitness.plot([], [], color='#f39c12', linewidth=2)
ax_mean_fitness.set_xlim(0, len(trait_snapshots)); ax_mean_fitness.set_ylim(0, 1)
ax_mean_fitness.set_xlabel('Generation', color='white')
ax_mean_fitness.set_ylabel('Mean Fitness', color='#f39c12')
ax_mean_fitness.set_title('Mean Fitness', color='white', fontsize=11)

plt.tight_layout()

def update_evolution(frame):
    traits = trait_snapshots[frame]
    counts, _ = np.histogram(traits, bins=trait_bins)
    for bar, height in zip(histogram_bars, counts):
        bar.set_height(height)
    optimum_line.set_xdata([optimum_log[frame], optimum_log[frame]])
    phase = 'stable' if frame < 10 else 'shifting'
    histogram_title.set_text(f'Gen {frame} — {phase}  N={pop_size_log[frame]}')
    fitness_line.set_data(range(frame + 1), mean_fitness_log[:frame + 1])
    return (*histogram_bars, optimum_line, histogram_title, fitness_line)

anim = FuncAnimation(fig, update_evolution, frames=len(trait_snapshots), interval=80, blit=True)
anim.save('evolution_adapt.gif', writer=PillowWriter(fps=14))
plt.close()

![Moving optimum adaptation](img/evolution_adapt.gif)

### Mutation Rate vs Changing Environment

Tradeoff between accumulating deleterious mutation in stable environments and adapting faster in changing environments

In [10]:
# Mutation rate tradeoff: stable vs. shifting environment

mutation_rates = [0.005, 0.02, 0.07, 0.15, 0.5]
mutation_rate_colors = ['#3498db', '#2ecc71', '#f1c40f', '#e67e22', '#e74c3c']
results_stable, results_shifting = [], []

for mut_rate in mutation_rates:
    sim = EvolutionSim(pop_size=200, mut_rate=mut_rate, mut_std=0.2)
    stable_fitness   = []
    shifting_fitness = []
    for _ in range(80):          # stable phase
        _, mean_fitness = sim.step()
        stable_fitness.append(mean_fitness)
    for _ in range(80):          # shifting phase
        _, mean_fitness = sim.step(shift=0.09)
        shifting_fitness.append(mean_fitness)
    results_stable.append(stable_fitness)
    results_shifting.append(shifting_fitness)

fig, (ax_stable, ax_shifting) = plt.subplots(1, 2, figsize=(12, 5), facecolor='#111')
for ax in (ax_stable, ax_shifting):
    ax.set_facecolor('#111')
    for spine in ax.spines.values(): spine.set_color('#444')
    ax.tick_params(colors='white')

for i, (mut_rate, color) in enumerate(zip(mutation_rates, mutation_rate_colors)):
    ax_stable.plot(results_stable[i],   color=color, linewidth=2, label=f'μ={mut_rate}')
    ax_shifting.plot(results_shifting[i], color=color, linewidth=2, label=f'μ={mut_rate}')

for ax, title in zip((ax_stable, ax_shifting), ['Stable Environment', 'Shifting Environment']):
    ax.set_xlabel('Generation', color='white')
    ax.set_ylabel('Mean Fitness', color='white')
    ax.set_title(title, color='white', fontsize=11)
    ax.set_ylim(0, 1)
    ax.legend(facecolor='#222', labelcolor='white', fontsize=9)
    ax.grid(True, alpha=0.15, color='#444')

plt.suptitle('The Mutation Rate Tradeoff', color='white', fontsize=13)
plt.tight_layout()
plt.savefig('mutation.png')
plt.close()

![Mutation tradeoff](img/mutation.png)

## 6. Evolutionary Ecology — Predator-Prey Coevolution

**Primary producers → Prey → Predators**

Fitness emerges purely from interactions between agents, unlike the previous example's arbitrary fitness function.

| Agent | What it does | Heritable trait |
|-------|-------------|-----------------|
| **Resource cells** (100×100 grid) | Slowly regrow each step; get eaten by prey | -- |
| **Prey** | Eat plants, run from nearby predators, reproduce when well-fed, starve on bare patches | Flee speed |
| **Predators** | Detect prey within a fixed radius, then chase; starve if unsuccessful | Chase speed |

**Energy:**
- Prey gain energy from eating, but moving fast burns more of it
- Predators gain energy from successful catches, but chasing fast also burns more energy
- Prey sharing a patch split the food — crowded patches mean less for everyone

**The Arms Race:**
- Faster prey are harder to catch → flee speed is selected for
- Faster predators catch prey more easily → chase speed is selected for
- But speed is metabolically expensive, faster individuals starve faster

In [41]:
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import Image as IPImage
from matplotlib.lines import Line2D

class PredPreySim:
    """
    Three-trophic-level ABM: Resources --> Prey --> Predators.

    Resource grid: 100x100 cells, each regenerating at res_regen/step.
    
    Prey in the same cell split the available resource equally (competition).

    Speed-dependent catch:  eff_prob = catch_prob x clip((pred_spd - prey_spd)/pred_spd, 0.05, 1)
        
    Speed-dependent metabolism:  metab = metab_base x speed^1.5

    Heritable traits:
      prey     --> flee_speed   (evolves up under predation)
      predator --> chase_speed  (evolves up as prey become faster)
    """
    W, H = 100, 100

    def __init__(self, n_prey=300, n_pred=100, seed=123):
        rng = np.random.default_rng(seed)

        # Resource layer
        self.resources = np.full((self.H, self.W), 0.2, dtype=np.float32)
        self.res_regen = 0.01   # per-cell per-step regrowth
        self.res_max   = 1.0
        self.eat_amt   = 0.5     # max resource one prey can eat per step
        self.eat_gain  = 2.0     # energy gained per unit resource consumed

        # Prey
        self.px       = rng.uniform(0, self.W, n_prey)
        self.py       = rng.uniform(0, self.H, n_prey)
        self.p_speed = np.clip(rng.normal(1.0, 0.3, n_prey), 0.3, None)
        self.p_energy = np.full(n_prey, 8.0)  

        # Predators
        self.qx       = rng.uniform(0, self.W, n_pred)
        self.qy       = rng.uniform(0, self.H, n_pred)
        self.q_speed = np.clip(rng.normal(1.1, 0.3, n_pred), 0.3, None)
        self.q_energy = np.full(n_pred, 10.0)
        self._rng = rng

        # Parameters
        self.prey_metab_base = 0.1   # prey metabolism: actual = base x speed^1.5
        self.prey_repro      = 5.0    # energy threshold to reproduce
        self.flee_rad        = 5.0    # flee predators within this radius

        self.sense_rad       = 5.0    # predator detection radius (fixed)
        self.catch_rad       = 1.5    # must be within this to attempt a catch
        self.catch_prob      = 0.9    # base catch probability (modified by speed gap)
        self.pred_metab_base = 0.5    # predator metabolism: actual = base x speed^1.5
        self.catch_gain      = 10.0   # energy gained per successful catch
        self.pred_repro      = 20.0   # energy threshold to reproduce

        self.mut_rate = 0.1
        self.mut_std  = 0.5

    def _toro(self, ax, ay, bx, by):
        """Toroidal (wrap-around) displacement vector."""
        return ((ax - bx + self.W/2) % self.W - self.W/2,
                (ay - by + self.H/2) % self.H - self.H/2)

    def step(self):
        W, H, rng = self.W, self.H, self._rng
        n_prey = len(self.px); n_pred = len(self.qx)

        # 0. Prey grazing: split resource equally among patch-mates
        if n_prey > 0:
            xi = np.clip(self.px.astype(int), 0, W-1)
            yi = np.clip(self.py.astype(int), 0, H-1)
            cell_counts = np.zeros((H, W), dtype=np.int32)
            np.add.at(cell_counts, (yi, xi), 1)
            competitors = cell_counts[yi, xi]
            available   = self.resources[yi, xi]
            share = np.minimum(self.eat_amt, available / np.maximum(competitors, 1))
            np.add.at(self.resources, (yi, xi), (-share).astype(np.float32))
            np.clip(self.resources, 0, None, out=self.resources)

            # Speed-dependent metabolism: faster prey burn more energy
            prey_metabolism = self.prey_metab_base * self.p_speed ** 1.5
            self.p_energy += share * self.eat_gain - prey_metabolism

        # Resources regenerate each step
        np.minimum(self.resources + self.res_regen, self.res_max, out=self.resources)

        # 1. Prey movement: flee nearest predator / random walk
        if n_prey > 0:
            if n_pred > 0:
                dx_fp, dy_fp = self._toro(self.qx[None,:], self.qy[None,:],
                                          self.px[:,None], self.py[:,None])
                d_fp         = np.hypot(dx_fp, dy_fp)
                nearest_q    = np.argmin(d_fp, axis=1)
                d_near       = d_fp[np.arange(n_prey), nearest_q]
                is_fleeing   = d_near < self.flee_rad
                away_dx      = -dx_fp[np.arange(n_prey), nearest_q]
                away_dy      = -dy_fp[np.arange(n_prey), nearest_q]
                flee_angle   = np.arctan2(away_dy, away_dx) + rng.normal(0, 0.25, n_prey)
                random_angle = rng.uniform(0, 2*np.pi, n_prey)
                prey_move_x  = np.where(is_fleeing, np.cos(flee_angle), np.cos(random_angle))
                prey_move_y  = np.where(is_fleeing, np.sin(flee_angle), np.sin(random_angle))
            else:
                random_angle = rng.uniform(0, 2*np.pi, n_prey)
                prey_move_x = np.cos(random_angle); prey_move_y = np.sin(random_angle)
            self.px = (self.px + self.p_speed * prey_move_x) % W
            self.py = (self.py + self.p_speed * prey_move_y) % H

        # 2. Prey starvation
        if n_prey > 0:
            prey_survived = self.p_energy > 0
            self.px       = self.px[prey_survived];      self.py       = self.py[prey_survived]
            self.p_speed  = self.p_speed[prey_survived]; self.p_energy = self.p_energy[prey_survived]
            n_prey = len(self.px)

        # 3. Predator movement: sense --> chase / random walk
        if n_pred > 0 and n_prey > 0:
            dx, dy   = self._toro(self.px[None,:], self.py[None,:],
                                  self.qx[:,None], self.qy[:,None])
            dists    = np.hypot(dx, dy)
            in_sense = dists < self.sense_rad
            prey_detected = in_sense.any(axis=1)
            sensed_distances = np.where(in_sense, dists, np.inf)
            nearest_prey_idx = np.argmin(sensed_distances, axis=1)
            cdx = dx[np.arange(n_pred), nearest_prey_idx]
            cdy = dy[np.arange(n_pred), nearest_prey_idx]
            cmag = np.maximum(np.hypot(cdx, cdy), 1e-6)
            random_angle_pred = rng.uniform(0, 2*np.pi, n_pred)
            pred_move_x = np.where(prey_detected, cdx/cmag, np.cos(random_angle_pred))
            pred_move_y = np.where(prey_detected, cdy/cmag, np.sin(random_angle_pred))
            # Each predator moves at its own heritable chase speed
            self.qx = (self.qx + self.q_speed * pred_move_x) % W
            self.qy = (self.qy + self.q_speed * pred_move_y) % H
            # Speed-dependent metabolism: faster predators burn more energy
            self.q_energy -= self.pred_metab_base * self.q_speed ** 1.5

            # 4. Catching prey: probability scales with predator speed advantage
            dx2, dy2 = self._toro(self.px[None,:], self.py[None,:],
                                  self.qx[:,None], self.qy[:,None])
            dists2 = np.hypot(dx2, dy2)
            prey_alive_mask = np.ones(n_prey, dtype=bool)
            for i in range(n_pred):
                catchable_idx = np.where((dists2[i] < self.catch_rad) & prey_alive_mask)[0]
                if not len(catchable_idx): continue
                # Faster predators relative to prey are more likely to catch
                speed_advantage = np.clip(
                    (self.q_speed[i] - self.p_speed[catchable_idx]) / max(self.q_speed[i], 1e-6),
                    0.05, 1.0)
                effective_catch_prob = self.catch_prob * speed_advantage
                caught_idx = catchable_idx[rng.random(len(catchable_idx)) < effective_catch_prob]
                if len(caught_idx):
                    prey_alive_mask[caught_idx[0]] = False
                    self.q_energy[i] += self.catch_gain
            self.px      = self.px[prey_alive_mask];      self.py      = self.py[prey_alive_mask]
            self.p_speed = self.p_speed[prey_alive_mask]; self.p_energy = self.p_energy[prey_alive_mask]
        else:
            self.q_energy -= self.pred_metab_base * self.q_speed ** 1.5

        # 5. Predator starvation
        pred_alive_mask = self.q_energy > 0
        self.qx      = self.qx[pred_alive_mask];      self.qy      = self.qy[pred_alive_mask]
        self.q_speed = self.q_speed[pred_alive_mask]; self.q_energy = self.q_energy[pred_alive_mask]

        # 6. Prey reproduction (asexual + mutation)
        n_prey = len(self.px)
        if n_prey > 0:
            prey_reproduce_mask = self.p_energy > self.prey_repro
            if prey_reproduce_mask.any():
                n_offspring = prey_reproduce_mask.sum()
                self.p_energy[prey_reproduce_mask] /= 2
                offspring_speed = self.p_speed[prey_reproduce_mask].copy()
                mutation_mask = rng.random(n_offspring) < self.mut_rate
                offspring_speed[mutation_mask] += rng.normal(0, self.mut_std, mutation_mask.sum())
                offspring_energy = self.p_energy[prey_reproduce_mask].copy()
                self.px       = np.append(self.px, (self.px[prey_reproduce_mask] + rng.normal(0, 0.5, n_offspring)) % W)
                self.py       = np.append(self.py, (self.py[prey_reproduce_mask] + rng.normal(0, 0.5, n_offspring)) % H)
                self.p_speed  = np.append(self.p_speed, np.clip(offspring_speed, 0.1, None))
                self.p_energy = np.append(self.p_energy, offspring_energy)

        # 7. Predator reproduction (asexual + mutation)
        pred_reproduce_mask = self.q_energy > self.pred_repro
        if pred_reproduce_mask.any():
            n_offspring = pred_reproduce_mask.sum()
            self.q_energy[pred_reproduce_mask] /= 2
            offspring_speed = self.q_speed[pred_reproduce_mask].copy()
            mutation_mask = rng.random(n_offspring) < self.mut_rate
            offspring_speed[mutation_mask] += rng.normal(0, self.mut_std, mutation_mask.sum())
            offspring_energy = self.q_energy[pred_reproduce_mask].copy()
            self.qx       = np.append(self.qx, (self.qx[pred_reproduce_mask] + rng.normal(0, 0.5, n_offspring)) % W)
            self.qy       = np.append(self.qy, (self.qy[pred_reproduce_mask] + rng.normal(0, 0.5, n_offspring)) % H)
            self.q_speed  = np.append(self.q_speed,  np.clip(offspring_speed, 0.5, None))
            self.q_energy = np.append(self.q_energy, offspring_energy)

        return len(self.px), len(self.qx)


# Run simulation
sim_pp = PredPreySim()
W, H = sim_pp.W, sim_pp.H
STEPS_PP = 1000
prey_positions, pred_positions, resource_snapshots = [], [], []
prey_count_log, pred_count_log, res_mean_log = [], [], []
prey_speed_log, pred_speed_log = [], []

for _ in range(STEPS_PP):
    n_prey_alive, n_pred_alive = sim_pp.step()
    prey_positions.append((sim_pp.px.copy(), sim_pp.py.copy()))
    pred_positions.append((sim_pp.qx.copy(), sim_pp.qy.copy()))
    resource_snapshots.append(sim_pp.resources.copy())
    prey_count_log.append(n_prey_alive); pred_count_log.append(n_pred_alive)
    res_mean_log.append(float(sim_pp.resources.mean()))
    prey_speed_log.append(float(sim_pp.p_speed.mean()) if n_prey_alive else float('nan'))
    pred_speed_log.append(float(sim_pp.q_speed.mean()) if n_pred_alive else float('nan'))

# Animation: resource heatmap + spatial view + live population panel
FRAME_SKIP = 2   # render every Nth simulation step to keep GIF size manageable
animation_frames = list(range(0, STEPS_PP, FRAME_SKIP))

fig, (ax_spatial, ax_population) = plt.subplots(1, 2, figsize=(13, 5.5), facecolor='#111')
ax_spatial.set_facecolor('#0a0a14'); ax_spatial.axis('off')
ax_spatial.set_xlim(0, W); ax_spatial.set_ylim(0, H)

resource_heatmap = ax_spatial.imshow(resource_snapshots[0], origin='lower', extent=[0, W, 0, H],
                       vmin=0, vmax=1, cmap='YlGn', alpha=0.55, zorder=1)
step_title = ax_spatial.set_title('Step 0', color='white', fontsize=11)
sc_prey = ax_spatial.scatter([], [], s=12,  c='#00008b', alpha=0.80, zorder=3)
sc_pred = ax_spatial.scatter([], [], s=90, c='#e74c3c', alpha=0.90, zorder=4, marker='^')
ax_spatial.legend(handles=[
    Line2D([0],[0], marker='s', color='w', markerfacecolor='#5dca6d', markersize=7, label='Resources'),
    Line2D([0],[0], marker='o', color='w', markerfacecolor='#00008b', markersize=6, label='Prey'),
    Line2D([0],[0], marker='^', color='w', markerfacecolor='#e74c3c', markersize=9, label='Predators'),
], facecolor='#222', labelcolor='white', fontsize=8, loc='upper right')

pop_y_max = max(max(prey_count_log) * 1.1, 200)
ax_population.set_facecolor('#111')
for sp in ax_population.spines.values(): sp.set_color('#444')
ax_population.tick_params(colors='white')
prey_line, = ax_population.plot([], [], color='#3498db', lw=2, label='Prey')
pred_line, = ax_population.plot([], [], color='#e74c3c', lw=2, label='Predators')
ax_population.set_xlim(0, STEPS_PP); ax_population.set_ylim(0, pop_y_max)
ax_population.set_xlabel('Step', color='white'); ax_population.set_ylabel('Population', color='white')

ax_resource = ax_population.twinx()
for sp in ax_resource.spines.values(): sp.set_color('#444')
ax_resource.spines['right'].set_color('#5dca6d')
ax_resource.tick_params(colors='#5dca6d')
ax_resource.set_ylim(0, 1.0)
ax_resource.set_ylabel('Mean resource', color='#5dca6d', fontsize=9)
resource_line, = ax_resource.plot([], [], color='#5dca6d', lw=1.5, alpha=0.7, label='Resources')

ax_population.set_title('Population & Resources', color='white', fontsize=11)
ax_population.legend([prey_line, pred_line, resource_line], ['Prey', 'Predators', 'Mean resource'],
              facecolor='#222', labelcolor='white', fontsize=9)
ax_population.grid(True, alpha=0.12, color='#444')
plt.tight_layout()

def update_predprey(frame_idx):
    i = animation_frames[frame_idx]
    px, py = prey_positions[i]; qx, qy = pred_positions[i]
    resource_heatmap.set_data(resource_snapshots[i])
    sc_prey.set_offsets(np.column_stack([px, py]) if len(px) else np.empty((0, 2)))
    sc_pred.set_offsets(np.column_stack([qx, qy]) if len(qx) else np.empty((0, 2)))
    step_title.set_text(f'Step {i+1}  ·  Prey: {prey_count_log[i]}  ·  Pred: {pred_count_log[i]}')
    prey_line.set_data(range(i+1), prey_count_log[:i+1])
    pred_line.set_data(range(i+1), pred_count_log[:i+1])
    resource_line.set_data(range(i+1), res_mean_log[:i+1])
    return sc_prey, sc_pred, step_title, prey_line, pred_line, resource_line

anim_predprey = FuncAnimation(fig, update_predprey, frames=len(animation_frames), interval=60, blit=False)
anim_predprey.save('predprey.gif', writer=PillowWriter(fps=18))
plt.close()


![Predator-Prey dynamics](img/predprey.gif)

In [40]:
# Analyze ecological model
step_range = np.arange(STEPS_PP)

fig = plt.figure(figsize=(15, 9), facecolor='#111')
grid_spec = fig.add_gridspec(2, 2, hspace=0.45, wspace=0.38)
ax_population    = fig.add_subplot(grid_spec[0, :])
ax_phase_portrait = fig.add_subplot(grid_spec[1, 0])
ax_arms_race     = fig.add_subplot(grid_spec[1, 1])

spine_color = '#444'

# Panel 1: Population + resource dynamics
ax_population.set_facecolor('#0d0d1a')
for sp in ax_population.spines.values(): sp.set_color(spine_color)
ax_population.tick_params(colors='white')

ax_population.plot(step_range, prey_count_log, color='#3498db', lw=2, label='Prey', alpha=0.9)
ax_population.plot(step_range, pred_count_log, color='#e74c3c', lw=2, label='Predators', alpha=0.9)
ax_population.set_xlim(0, STEPS_PP)
ax_population.set_ylim(0, max(max(prey_count_log)*1.05, 10))
ax_population.set_xlabel('Step', color='white')
ax_population.set_ylabel('Population', color='white')
ax_population.set_title('Population Dynamics', color='white', fontsize=12)
ax_population.grid(True, alpha=0.12, color=spine_color)

ax_resource_axis = ax_population.twinx()
ax_resource_axis.set_ylim(0, 1.05)
ax_resource_axis.set_ylabel('Mean resource level', color='#5dca6d', fontsize=9)
ax_resource_axis.tick_params(colors='#5dca6d')
for sp in ax_resource_axis.spines.values(): sp.set_color(spine_color)
ax_resource_axis.spines['right'].set_color('#5dca6d')
ax_resource_axis.plot(step_range, res_mean_log, color='#5dca6d', lw=1.5, alpha=0.7, label='Resource')

population_legend_handles = [plt.Line2D([0],[0], color='#3498db', lw=2),
          plt.Line2D([0],[0], color='#e74c3c', lw=2),
          plt.Line2D([0],[0], color='#5dca6d', lw=1.5)]
ax_population.legend(population_legend_handles, ['Prey', 'Predators', 'Mean resource'],
              facecolor='#222', labelcolor='white', fontsize=9, loc='upper right')

# Panel 2: Phase portrait
ax_phase_portrait.set_facecolor('#0d0d1a')
for sp in ax_phase_portrait.spines.values(): sp.set_color(spine_color)
ax_phase_portrait.tick_params(colors='white')

time_colormap = plt.cm.plasma
time_gradient = time_colormap(np.linspace(0, 1, STEPS_PP))
for i in range(STEPS_PP - 1):
    ax_phase_portrait.plot(prey_count_log[i:i+2], pred_count_log[i:i+2], color=time_gradient[i], lw=1.2, alpha=0.7)

ax_phase_portrait.scatter([prey_count_log[0]], [pred_count_log[0]], s=80, c='white', zorder=5, marker='o')
ax_phase_portrait.scatter([prey_count_log[-1]], [pred_count_log[-1]], s=80, c='yellow', zorder=5, marker='*')
ax_phase_portrait.text(prey_count_log[0]+3, pred_count_log[0]+1, 'start', color='white', fontsize=8)
ax_phase_portrait.text(prey_count_log[-1]+3, pred_count_log[-1]+1, 'end', color='yellow', fontsize=8)

colorbar_source = plt.cm.ScalarMappable(cmap=time_colormap, norm=plt.Normalize(0, STEPS_PP))
colorbar_source.set_array([])
colorbar = plt.colorbar(colorbar_source, ax=ax_phase_portrait)
colorbar.set_label('Step', color='white', fontsize=9)
colorbar.ax.yaxis.set_tick_params(color='white')
plt.setp(colorbar.ax.yaxis.get_ticklabels(), color='white')

ax_phase_portrait.set_xlabel('Prey population', color='white')
ax_phase_portrait.set_ylabel('Predator population', color='white')
ax_phase_portrait.set_title('Phase Portrait', color='white', fontsize=12)
ax_phase_portrait.grid(True, alpha=0.12, color=spine_color)

# Panel 3: Coevolutionary arms race
ax_arms_race.set_facecolor('#0d0d1a')
for sp in ax_arms_race.spines.values(): sp.set_color(spine_color)
ax_arms_race.tick_params(colors='white')

prey_mean_speed = np.array(prey_speed_log, dtype=float)
pred_mean_speed = np.array(pred_speed_log, dtype=float)

ax_arms_race.plot(step_range, prey_mean_speed, color='#3498db', lw=2, label='Prey flee speed')
ax_arms_race.plot(step_range, pred_mean_speed, color='#e74c3c', lw=2, label='Predator chase speed')
ax_arms_race.set_xlabel('Step', color='white')
ax_arms_race.set_ylabel('Mean speed', color='white')
ax_arms_race.tick_params(axis='y', colors='white')
ax_arms_race.set_xlim(0, STEPS_PP)
ax_arms_race.set_title('Coevolutionary Arms Race', color='white', fontsize=12)
ax_arms_race.legend(facecolor='#222', labelcolor='white', fontsize=9)
ax_arms_race.grid(True, alpha=0.12, color=spine_color)

plt.savefig('predprey_analysis.png', dpi=110, bbox_inches='tight',
            facecolor='#111', edgecolor='none')
plt.close()


![Predator-Prey analysis](img/predprey_analysis.png)

---

## Summary: Agent Based Models

<div>
<img src="img/predation.jpg" width="700"/>
</div>

Lynx carrying snowshoe hare photo by Jeff Lepore.

| Application | Example Seen | Core Result |
|-------------|--------------|-----------|
| **Artifical life** | Game of Life, Boids | Emergent complex behaviour from local rules |
| **Epidemiology** | Spatial SIR models | Spatial heterogeneity influences epidemic dynamics |
| **Evolutionary Genetics** | Moving optimum model | Populations continuously adapt to changing environments or go extinct |
| **Evolutionary Ecology** | Predatory-prey model | Population cycles, evolutionary arms race |

---

### Drawbacks of ABMs in natural sciences
- **Combinatorial explosion**: number of parameters defining a model increases rapidly with model complexity
- **Computational cost**: complicated ABMs can take a long time to compute, but techniques such as parallelization are well suited to these problems.
- **Lack of analytical insight**: trades mathematical rigour for dozens of time-series plots
- *Easy to produce very complex and impressive simulations that don't actually teach us a whole lot or don't represent the system we are trying to study very well*

In essence, agent-based models are very easy to apply incorrectly and shouldn't be used when analytical results are likely available.

However, they are extremely flexible and can provide basic intuition into a complex system and verify analytical results when empirical studies aren't possible.

---

### Recommended Reading/Sources

- *A Biologist's Guide to Mathematical Modeling in Ecology and Evolution* - Otto & Day
- *Modeling Evolution: An introduction to numerical methods* - Roff
- *SLiM 4: Multispecies Eco-Evolutionary Modeling* - Haller & Messer
- *Nonlinear Dynamics and Chaos: With Applications to Physics, Biology, Chemistry, and Engineering* - Strogatz
- *Emergent Garden* — https://emergentgarden.io/

---

## Thanks for listening!

The full Jupyter notebook can be found at https://github.com/thomas-wildeboer/ABM-dynamical-systems-lecture